In [ ]:
!pip install ultralytics -q
import cv2
import numpy as np
from ultralytics import YOLO

model = YOLO('best.pt').to('cuda') # Nhớ trỏ đúng model best của bạn nhé

# Đọc video
cap = cv2.VideoCapture("video.mp4")

# Lấy kích thước tự động (Chống lệch Polygon)
w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = int(cap.get(cv2.CAP_PROP_FPS))

# ÉP XUẤT THẲNG RA MP4
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
video_writer = cv2.VideoWriter("ket_qua.mp4", fourcc, fps, (w, h))

def get_center(box):
    cx = int((box[0] + box[2]) / 2)
    chieu_cao_box = box[3] - box[1]
    cy = int(box[3] - (chieu_cao_box * 0.15))
    # TRẢ VỀ THÊM CHIỀU CAO
    return (cx, cy, chieu_cao_box)

# Thêm tham số chieu_cao_box_hien_tai vào hàm
def count(id_xe, diem_hien_tai, diem_cu, lan_duong, in_counts, out_counts, danh_sach, frame_hien_tai, chieu_cao_box_hien_tai):
    cx, cy = diem_hien_tai
    for i, lan in enumerate(lan_duong):
        if cv2.pointPolygonTest(lan, diem_hien_tai, False) >= 0:
            cy_hien_tai = diem_hien_tai[1]
            cy_cu = diem_cu[1]

            if cy_hien_tai != cy_cu:
                is_ghost = False
                # Nhận thêm old_height từ bộ nhớ
                for old_id, (old_cx, old_cy, old_height, old_frame) in vi_tri_xe_da_dem.items():
                    frame_lech = frame_hien_tai - old_frame
                    
                    # 1. NỚI RỘNG THỜI GIAN NHỚ LÊN 20 FRAME
                    # Giúp tóm gọn các xe đi chậm bị chớp tắt ID muộn
                    if frame_lech < 20: 
                        khoang_cach = np.sqrt((cx - old_cx)**2 + (cy - old_cy)**2)
                        
                        # 2. CÔNG THỨC KẸP BIÊN THẮT CHẶT
                        # Ép đáy (30px): Đảm bảo xe nhỏ tít đằng xa vẫn có lưới bắt ma
                        # Tỷ lệ (0.5): Chỉ lấy 50% chiều cao thân xe thay vì 80% như cũ
                        # Ép trần (70px): Khóa chặt bán kính tối đa. Dù xe có to như cái nhà, hố đen cũng không vượt quá 70px.
                        ban_kinh_bat_ma = max(30, min(old_height * 0.5, 70))
                        
                        if khoang_cach < ban_kinh_bat_ma:
                            is_ghost = True 
                            break

                if not is_ghost:
                    if cy_hien_tai > cy_cu:
                        in_counts[i] += 1
                    elif cy_hien_tai < cy_cu:
                        out_counts[i] += 1

                    # Lưu lại vết làm mốc
                    vi_tri_xe_da_dem[id_xe] = (cx, cy, chieu_cao_box_hien_tai, frame_hien_tai)

                danh_sach.add(id_xe)
            break

def draw(frame, lan_duong, in_counts, out_counts):
    for i, lan in enumerate(lan_duong):
        cv2.polylines(frame, [lan], isClosed=True, color=(255, 0, 255), thickness=2)
        xs = [pt[0] for pt in lan]
        ys = [pt[1] for pt in lan]
        cx = int(sum(xs) / len(xs))
        cy = int(sum(ys) / len(ys))
        tong_xe = str(in_counts[i] + out_counts[i])
        cv2.putText(
            frame, tong_xe, org=(cx, cy), fontFace=cv2.FONT_HERSHEY_SIMPLEX,
            fontScale=1.5, color=(255, 255, 255), thickness=5
        )
    return frame

lan_duong = [
    np.array([[832, 679], [919, 705], [568, 957], [428, 907]]),
    np.array([[919, 707], [1026, 735], [740, 1021], [566, 955]]),
    np.array([[738, 1015], [1028, 735], [1140, 759], [884, 1069], [742, 1021]]),
    np.array([[1138, 757], [1286, 783], [1024, 1071], [884, 1068]]),
    np.array([[205, 746], [142, 714], [622, 572], [669, 593]]),
    np.array([[145, 712], [110, 683], [581, 559], [622, 572]]),
    np.array([[106, 684], [88, 656], [549, 548], [581, 557]]),
    np.array([[89, 651], [69, 621], [525, 539], [547, 548]])
]

in_counts = [0] * len(lan_duong)
out_counts = [0] * len(lan_duong)
danh_sach_xe_da_dem = set()
toa_do_cu = {}

# CHUYỂN THÀNH DICTIONARY ĐỂ THEO DÕI VỊ TRÍ ĐỘNG CỦA TỪNG ID
vi_tri_xe_da_dem = {}
frame_count = 0

print("🚀 Đang chạy mô hình AI và lưu thẳng ra file... Tốc độ bàn thờ luôn!")

while cap.isOpened():
    success, frame = cap.read()
    if not success:
        break

    frame_count += 1

    # Dọn dẹp bộ theo dõi định kỳ để giải phóng bộ nhớ
    vi_tri_xe_da_dem = {k: v for k, v in vi_tri_xe_da_dem.items() if frame_count - v[3] < 60}

    # Quét YOLO (Mở rộng classes 4, 5, 6, 7)
    results = model.track(frame, persist=True, classes=[4, 5, 6, 7], conf=0.25, tracker="bytetrack.yaml", verbose=False)
    frame = results[0].plot()

    if results[0].boxes.id is not None:
        boxes = results[0].boxes.xyxy.cpu().numpy()
        ids = results[0].boxes.id.cpu().numpy().astype(int)

        for box, v_id in zip(boxes, ids):
            # Nhận 3 giá trị từ hàm
            cx, cy, chieu_cao_box = get_center(box)
            diem_hien_tai = (cx, cy)
            diem_cu = toa_do_cu.get(v_id, diem_hien_tai)
            toa_do_cu[v_id] = diem_hien_tai

            if v_id in danh_sach_xe_da_dem:
                # LƯU THÊM CHIỀU CAO VÀO BỘ NHỚ
                vi_tri_xe_da_dem[v_id] = (cx, cy, chieu_cao_box, frame_count)
                continue

            # Đẩy thêm chieu_cao_box vào tham số của hàm count
            count(v_id, diem_hien_tai, diem_cu, lan_duong, in_counts, out_counts, danh_sach_xe_da_dem, frame_count, chieu_cao_box)

    frame = draw(frame, lan_duong, in_counts, out_counts)
    video_writer.write(frame)

cap.release()
video_writer.release()

print("✅ Xong! Quá nhanh quá nguy hiểm.")
print("👉 Hãy nhìn sang cột Files bên trái Colab, tải file 'ket_qua.mp4' về nhé!")

🚀 Đang chạy mô hình AI và lưu thẳng ra file
✅ Xong
👉 Download file 'ket_qua.mp4'
